# Uncertainty Analysis of a Bending Moment

In this example, we demonstrate how to perform an uncertainty analysis of the maximum bending moment in a sheet pile wall.

We consider a sheet pile wall model implemented in Kratos GeoMechanicsApplication, corresponding to the CROW sheet pile reference case. In this example, the Kratos model uses a linear-elastic soil model.

The objective is to quantify the effect of uncertainty in the soil parameters on the maximum bending moment of the sheet pile wall. The uncertainty analysis is performed using the [probabilistic library](https://deltares.github.io/ProbabilisticLibrary/) developed by Deltares.

We start by importing the required classes and packages:

In [ ]:
import KratosMultiphysics as Kratos
import KratosMultiphysics.GeoMechanicsApplication.prob_analysis as prob_analysis
import KratosMultiphysics.GeoMechanicsApplication.test_helper as test_helper
import KratosMultiphysics.StructuralMechanicsApplication as kratos_struct

import numpy as np
from probabilistic_library import UncertaintyProject, UncertaintyMethod, DistributionType
from pathlib import Path

We use the following method to locate the Kratos model:

In [ ]:
def find_kratos_crow_project_path():
 
    project_dir = Path("KratosCROW_2Stage_LinearElastic")
    candidates = [
        Path.cwd() / project_dir,
        Path.cwd() / "kratos-docs" / "docs" / "examples" / "crow_case" / project_dir,
    ]
 
    for parent in Path.cwd().resolve().parents:
        candidates.extend([
            parent / project_dir,
            parent / "kratos-docs" / "docs" / "examples" / "crow_case" / project_dir,
        ])
 
    for candidate in candidates:
        if candidate.is_dir():
            return candidate.resolve()
 
    raise FileNotFoundError(f"Could not find {project_dir} from {Path.cwd()}")
 
KRATOS_CROW_PROJECT_PATH = find_kratos_crow_project_path()

The following class runs a Kratos simulation of a sheet pile wall. The input parameters include the soil properties — Young’s modulus and Poisson’s ratio — as well as the selected sheet pile wall profile. The following sheet pile profiles are considered in this example:
* AZ18
* AZ18/10-10
* AZ26

The sheet pile wall properties are represented using equivalent material parameters, including wall thickness, Young’s modulus, and density. These equivalent values are derived from the moment of inertia $I$ and the cross-sectional area $A$, which are available for each sheet pile section (see [ArcelorMittal](https://sheetpiling.arcelormittal.com/products)).

The class returns the maximum bending moment in the sheet pile wall as output.

In [ ]:
class KratosCROW2StageLinearElastic():

    def __init__(self, layers, damwand_profile):

        self.layers = layers
        self.damwand_profile = damwand_profile

        self.template_project_path = KRATOS_CROW_PROJECT_PATH

        self.project_file_names = [
            "1_Initial_stage.json",
            "2_Final_equilibrium.json",
        ]

    def run_calculations(self, 
                        Excavated_Soil_1_young_modulus, 
                        Excavated_Soil_2_young_modulus,
                        Excavated_Soil_3_young_modulus,
                        Soil_Below_Pit_young_modulus,
                        Soil_Bottom_Left_young_modulus,
                        Soil_Bottom_Right_young_modulus,
                        Soil_Middle_Right_young_modulus,
                        Soil_Top_Right_young_modulus,
                        Excavated_Soil_1_poisson_ratio,
                        Excavated_Soil_2_poisson_ratio,
                        Excavated_Soil_3_poisson_ratio,
                        Soil_Below_Pit_poisson_ratio,
                        Soil_Bottom_Left_poisson_ratio,
                        Soil_Bottom_Right_poisson_ratio,
                        Soil_Middle_Right_poisson_ratio,
                        Soil_Top_Right_poisson_ratio):

        layer_young_modulus = [Excavated_Soil_1_young_modulus, 
                               Excavated_Soil_2_young_modulus,
                               Excavated_Soil_3_young_modulus,
                               Soil_Below_Pit_young_modulus,
                               Soil_Bottom_Left_young_modulus,
                               Soil_Bottom_Right_young_modulus,
                               Soil_Middle_Right_young_modulus,
                               Soil_Top_Right_young_modulus]
        
        layer_poisson_ratio = [Excavated_Soil_1_poisson_ratio,
                               Excavated_Soil_2_poisson_ratio,
                               Excavated_Soil_3_poisson_ratio,
                               Soil_Below_Pit_poisson_ratio,
                               Soil_Bottom_Left_poisson_ratio,
                               Soil_Bottom_Right_poisson_ratio,
                               Soil_Middle_Right_poisson_ratio,
                               Soil_Top_Right_poisson_ratio]

        input_parameters = []
        input_values = []

        # Young's modulus for each soil layer
        for layer, val in zip(self.layers, layer_young_modulus):
            input_parameters.append([0, f"PorousDomain.{layer}", Kratos.YOUNG_MODULUS])
            input_values.append(val)

        # Poisson's ratio for each soil layer
        for layer, val in zip(self.layers, layer_poisson_ratio):
            input_parameters.append([0, f"PorousDomain.{layer}", Kratos.POISSON_RATIO])
            input_values.append(val)

        beam_thickness, beam_young_modulus, beam_density = self._equivalent_values()

        # Beam thickness
        input_parameters.append([1, "PorousDomain.Beam", Kratos.THICKNESS])
        input_values.append(beam_thickness)

        # Beam young_modulus
        input_parameters.append([1, "PorousDomain.Beam", Kratos.YOUNG_MODULUS])
        input_values.append(beam_young_modulus)

        # Beam density
        input_parameters.append([1, "PorousDomain.Beam", Kratos.DENSITY])
        input_values.append(beam_density)

        # Format of a single 'output_parameters'  entry =
        # [<stage_nr>, 
        #  <function you want to perform on the results (can be None)>, 
        #  <get-function>, 
        #  <ModelPartName>, 
        #  <Variable>, 
        #  <node_id (can be None>)]
        output_parameters = [
             [
                 1,
                 np.max,
                 test_helper.get_nodal_variable,
                 "PorousDomain.Beam",
                 kratos_struct.BENDING_MOMENT,
                 None,
             ]
         ]

        try:
            prob_analysis_instance = prob_analysis.prob_analysis(
                self.template_project_path,
                input_parameters,
                output_parameters,
                self.project_file_names,
            )
            output_values = prob_analysis_instance.calculate(input_values)
            max_bending_moment = output_values[0]
        except Exception as e:
            print("An error occurred:", e)
            max_bending_moment = np.nan
        finally:
            # Clean up and remove the temporary folder
            prob_analysis_instance.finalize()

        value = max_bending_moment
        
        return value
    
    def _equivalent_values(self):

        if self.damwand_profile == 'AZ18': # see ArcelorMittal
            I = 3.420*10**(-4)
            A = 1.5*10**(-2)

        elif self.damwand_profile == 'AZ18_10_10': # see ArcelorMittal
            I = 3.554*10**(-4)
            A = 1.57*10**(-2)

        elif self.damwand_profile == 'AZ26': # see ArcelorMittal
            I = 5.551*10**(-4)
            A = 1.980*10**(-2)

        weight = A * 7850.0 # value for steel density in kg/m^3

        E_s = 210000000000
        EI = E_s * I
        EA = E_s * A
        b = 1.0
        g = 9.81

        t = np.sqrt(12*EI/EA)
        E = EA/(t*b)
        density = weight/(t*b)

        return t, E, density

The goal of the uncertainty analysis is to analyse the effect of uncertainty in the soil parameters on the maximum bending moment of a sheet pile wall. The uncertainty in the soil parameters is expressed using distribution functions, which are presented in the following tables. In this case, all distributions are log-normal with a coefficient of variation equal to $0.1$. The mean values of the input parameters differ per soil material (sand and clay).


##### Young's modulus per layer

| **Soil Layer**    | **Soil Material** | **Young’s Modulus, E (Pa)** | **CoV (-)** | **Distribution** |
|:------------------|------------------:|----------------------------:|------------:|:-----------------|
| Excavated Soil 1  | Clay              | 9.0 × 10⁵                   | 0.10        | Lognormal        |
| Excavated Soil 2  | Clay              | 9.0 × 10⁵                   | 0.10        | Lognormal        |
| Excavated Soil 3  | Clay              | 9.0 × 10⁵                   | 0.10        | Lognormal        |
| Soil Below Pit    | Clay              | 9.0 × 10⁵                   | 0.10        | Lognormal        |
| Middle Soil Right | Clay              | 9.0 × 10⁵                   | 0.10        | Lognormal        |
| Top Soil Right    | Clay              | 9.0 × 10⁵                   | 0.10        | Lognormal        |
| Bottom Soil Left  | Sand              | 7.4286 × 10⁶                | 0.10        | Lognormal        |
| Bottom Soil Right | Sand              | 7.4286 × 10⁶                | 0.10        | Lognormal        |
       
##### Poisson's ratio per layer

| **Soil Material** | **Mean Value (-)** | **CoV (-)** | **Distribution** |
|:------------------|-------------------:|------------:|:-----------------|
| Clay              | 0.20               | 0.10        | Lognormal        |
| Sand              | 0.30               | 0.10        | Lognormal        |

We now need to convert these uncertainty distributions into variables that can be interpreted by the probabilistic library. This is done as follows:

In [ ]:
layers = ["Excavated_Soil_1",
              "Excavated_Soil_2",
              "Excavated_Soil_3",
              "Soil_Below_Pit",
              "Soil_Bottom_Left",
              "Soil_Bottom_Right",
              "Soil_Middle_Right",
              "Soil_Top_Right"]

In [ ]:
def define_project_variables(project, layers):

    # define Young's modulus for each soil layer
    for layer in layers:
        if layer in ["Soil_Bottom_Left", "Soil_Bottom_Right"]:
            project.variables[f"{layer}_young_modulus"].distribution = DistributionType.log_normal
            project.variables[f"{layer}_young_modulus"].mean = 7.4286e+06
            project.variables[f"{layer}_young_modulus"].variation = 0.1
        else:
            project.variables[f"{layer}_young_modulus"].distribution = DistributionType.log_normal
            project.variables[f"{layer}_young_modulus"].mean = 9.0e+05
            project.variables[f"{layer}_young_modulus"].variation = 0.1

    # define Poisson's ratio for each soil layer
    for layer in layers:
        if layer in ["Soil_Bottom_Left", "Soil_Bottom_Right"]:
            project.variables[f"{layer}_poisson_ratio"].distribution = DistributionType.log_normal
            project.variables[f"{layer}_poisson_ratio"].mean = 0.3
            project.variables[f"{layer}_poisson_ratio"].variation = 0.1
        else:
            project.variables[f"{layer}_poisson_ratio"].distribution = DistributionType.log_normal
            project.variables[f"{layer}_poisson_ratio"].mean = 0.2
            project.variables[f"{layer}_poisson_ratio"].variation = 0.1
    
    return project

### AZ18

Now, let's perform the uncertainty analysis for the `AZ18` profile. 

We define a project using the UncertaintyProject class (step 1), connect the Kratos model to the uncertainty project (step 2), and define the project variables (step 3).

In [ ]:
uncertainty_wall = UncertaintyProject() # step (1)
uncertainty_wall.model = KratosCROW2StageLinearElastic(layers, 'AZ18').run_calculations # step (2)
uncertainty_wall = define_project_variables(uncertainty_wall, layers) # step (3)

We choose a crude Monte Carlo simulation method and consider between $15$ and $20$ samples:

In [ ]:
uncertainty_wall.settings.uncertainty_method = UncertaintyMethod.crude_monte_carlo
uncertainty_wall.settings.minimum_samples = 15
uncertainty_wall.settings.maximum_samples = 20

We run the uncertainty analysis using the `run()` method and plot the corresponding results. The results represent the uncertainty in the bending moment as a result of uncertainty in the soil input parameters.

In [ ]:
uncertainty_wall.run()
uncertainty_wall.stochast.plot()

### AZ18/10-10

We repeat the uncertainty analysis for the `AZ18/10-10` sheet pile wall profile:

In [ ]:
uncertainty_wall = UncertaintyProject()
uncertainty_wall.model = KratosCROW2StageLinearElastic(layers, 'AZ18_10_10').run_calculations
uncertainty_wall = define_project_variables(uncertainty_wall, layers)

uncertainty_wall.settings.uncertainty_method = UncertaintyMethod.crude_monte_carlo
uncertainty_wall.settings.minimum_samples = 15
uncertainty_wall.settings.maximum_samples = 20

uncertainty_wall.run()
uncertainty_wall.stochast.plot()

### AZ26

And finally, we run the uncertainty analysis for the `AZ26` profile:

In [ ]:
uncertainty_wall = UncertaintyProject()
uncertainty_wall.model = KratosCROW2StageLinearElastic(layers, 'AZ26').run_calculations
uncertainty_wall = define_project_variables(uncertainty_wall, layers)

uncertainty_wall.settings.uncertainty_method = UncertaintyMethod.crude_monte_carlo
uncertainty_wall.settings.minimum_samples = 15
uncertainty_wall.settings.maximum_samples = 20

uncertainty_wall.run()
uncertainty_wall.stochast.plot()